In [1]:
import pandas as pd
import numpy as np
import glob
import os

In [2]:
so2_files = glob.glob("../../data/raw/SO2_*.csv")
print("Files found:", so2_files)

Files found: ['../../data/raw\\SO2_2020.csv', '../../data/raw\\SO2_2021.csv', '../../data/raw\\SO2_2022.csv', '../../data/raw\\SO2_2023.csv']


In [3]:
so2_list = []
for f in so2_files:
    df = pd.read_csv(f, skiprows=7, low_memory=False, encoding="utf-8-sig")
    so2_list.append(df)
    print("Loaded:", f, df.shape)

so2_raw = pd.concat(so2_list, ignore_index=True)
print("SO2 combined shape:", so2_raw.shape)

Loaded: ../../data/raw\SO2_2020.csv (52338, 31)
Loaded: ../../data/raw\SO2_2021.csv (53290, 31)
Loaded: ../../data/raw\SO2_2022.csv (52925, 31)
Loaded: ../../data/raw\SO2_2023.csv (51465, 31)
SO2 combined shape: (210018, 31)


In [4]:
so2_raw.columns = [c.split("/")[0].strip() for c in so2_raw.columns]
print("Columns sample:", so2_raw.columns[:10].tolist())

Columns sample: ['Pollutant', 'NAPS ID', 'City', 'Province', 'Latitude', 'Longitude', 'Date', 'H01', 'H02', 'H03']


In [6]:
hour_cols = [c for c in so2_raw.columns if c.startswith("H")]
print("Hourly columns found:", len(hour_cols), "| Example:", hour_cols[:3], "...", hour_cols[-3:])

so2_raw[hour_cols] = so2_raw[hour_cols].replace(-999, np.nan)
so2_raw["SO2_daily"] = so2_raw[hour_cols].mean(axis=1, skipna=True)
print("Missing SO2_daily:", so2_raw["SO2_daily"].isna().sum())


Hourly columns found: 24 | Example: ['H01', 'H02', 'H03'] ... ['H22', 'H23', 'H24']
Missing SO2_daily: 6975


In [7]:
so2_raw["Date"] = pd.to_datetime(so2_raw["Date"], errors="coerce")
print("Unparsed Date rows (NaT):", so2_raw["Date"].isna().sum())
print("Date range:", so2_raw["Date"].min(), "→", so2_raw["Date"].max())

Unparsed Date rows (NaT): 0
Date range: 2020-01-01 00:00:00 → 2023-12-31 00:00:00


In [8]:
so2_cityday = so2_raw[["City", "Date", "SO2_daily"]].copy()
so2_cityday["Year"] = so2_cityday["Date"].dt.year
so2_cityday["Month"] = so2_cityday["Date"].dt.month

In [9]:
so2_cityday = so2_cityday.dropna(subset=["SO2_daily", "Date"])
so2_cityday = so2_cityday.drop_duplicates(subset=["City", "Date"])
print("Final shape:", so2_cityday.shape)
print("Year counts:\n", so2_cityday["Year"].value_counts().sort_index())

Final shape: (148454, 5)
Year counts:
 Year
2020    36908
2021    38057
2022    37618
2023    35871
Name: count, dtype: int64


In [10]:
os.makedirs("../../data/validated", exist_ok=True)
so2_cityday.to_csv("../../data/validated/SO2_cityday.csv", index=False)
print("Saved: SO2_cityday.csv")
so2_cityday.head()

Saved: SO2_cityday.csv


,City,Date,SO2_daily,Year,Month
0,St. John's,2020-01-01,0.066667,2020,1
1,St. John's,2020-01-02,0.458333,2020,1
2,St. John's,2020-01-03,0.470833,2020,1
3,St. John's,2020-01-04,0.766667,2020,1
4,St. John's,2020-01-05,0.666667,2020,1
